In [0]:
-- ============================================
-- SILVER 层: 数据清洗和转换
-- ============================================

-- 1. 处理客户维度 (SCD Type 2)
MERGE INTO aws3.silver.dim_user AS target
USING (
    WITH latest_customers AS (
        SELECT 
            id          ,
            login_name            ,
            nick_name             ,
            passwd                ,
            name                  ,
            phone_num             ,
            email                 ,
            head_img              ,
            user_level            ,
            birthday              ,
            gender                ,
            create_time           ,
            operate_time          ,
            status                ,
            ROW_NUMBER() OVER (PARTITION BY id ORDER BY operate_time DESC) as rn
        FROM aws3.bronze.user_info
        WHERE DATE(_bronze_load_ts) >= DATE_ADD(CURRENT_DATE(), -7)
        QUALIFY rn = 1
    )
    SELECT 
        id          ,
        login_name            ,
        nick_name             ,
        passwd                ,
        name                  ,
        phone_num             ,
        email                 ,
        head_img              ,
        user_level            ,
        birthday              ,
        gender                ,
        create_time,
        operate_time,
        status          ,
        operate_time as valid_from,
        LEAD(operate_time, 1) OVER (PARTITION BY id ORDER BY operate_time) as valid_to,
        'mysql_production' as source_system,
        UUID() as load_batch_id
    FROM latest_customers
) AS source
ON target.id = source.id 
   AND target.is_current = true
   AND MD5(CONCAT(target.id, target.name, target.phone_num, target.email,target.user_level)) <> md5(CONCAT(source.id, source.name, source.phone_num, source.email,source.user_level))
WHEN MATCHED THEN
    UPDATE SET 
        target.valid_to = source.valid_from,
        target.is_current = false,
        target.operate_time = CURRENT_TIMESTAMP()--是当前时间还是operate_time as valid_from?
WHEN NOT MATCHED THEN
INSERT (
    id          ,
    login_name            ,
    nick_name             ,
    passwd                ,
    name                  ,
    phone_num             ,
    email                 ,
    head_img              ,
    user_level            ,
    birthday              ,
    gender                ,
    create_time,
    operate_time,
    status          ,
    valid_from,
    valid_to,
    is_current,
    source_system,
    load_batch_id
)
VALUES (
    source.id          ,
    source.login_name            ,
    source.nick_name             ,
    source.passwd                ,
    source.name                  ,
    source.phone_num             ,
    source.email                 ,
    source.head_img              ,
    source.user_level            ,
    source.birthday              ,
    source.gender                ,
    source.create_time,
    source.operate_time,
    source.status   ,
    source.valid_from,
    NULL,
    true,
    source.source_system,
    source.load_batch_id
);
